# Transformer Block
At this point, we can start building a transformer using the `torch.nn` module. We will build a transformer block that represents the encoder part of the transformer model as illustrated below.

![transformer-block](./resources/transformer-block.png)

In [ ]:
from torch import nn

class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        d_ff: int = None,
        dropout: float = 0.1
    ):
        super().__init__()
        # The same as the attention that we talked about before
        # but pytorch has it ready and we don't need to implement
        # on our own
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm((d_model, ))
        # A convention is to have 4 * d_model as the output shape
        d_ff = d_model * 4 if not d_ff else d_ff
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm((d_model, ))
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, X):
        """X has shape (batch_size, sequence_length, d_model)"""
        # Apply multi-head attention to get new representation
        attn_output, _ = self.attention(X, X, X)
        # Add & Norm for the multi-head attention output
        norm1_input = X + attn_output
        norm1_output = self.norm1(norm1_input)
        # Feed forward
        ffn_output = self.linear_relu_stack(norm1_output)
        ffn_output = self.dropout2(ffn_output)
        # Add & Norm for the FFN output
        norm2_input = norm1_output + ffn_output
        norm2_output = self.norm2(norm2_input)
        return norm2_output

TransformerBlock(d_model=64, heads=8)

TransformerBlock(
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
  )
  (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
  )
  (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (dropout2): Dropout(p=0.1, inplace=False)
)

The above code simply put together what we have learned so far. In order the build and train a full transformer model, we will need more:
1. The embeddings layer
2. Stacking the transformer blocks
3. Output layer, to actually output something that is understandable for human
4. Some training data